# Funzioni Lambda  
In questo file sono contenute tutte le funzioni Lambda già implementate e caricate su AWS Lambda, con le opportune risorse dell'AWS API gateway configurate, e dunque pronte per l'utilizzo tramite API. Per indicazioni sul loro uso riferirisri alla documentazione

# Equation solver

In [1]:
import json
import sympy as sp

def solve_equation(event, context):
    res = {}
    res['headers'] = {"Access-Control-Allow-Origin" : "*"}
    try:
        try:
            eq = event['queryStringParameters']['eq']
            un = event['queryStringParameters']['un']
            latex = event['queryStringParameters']['latex']
            if latex == "false" or latex == "False" or latex == "0":
                latex = False
            else:
                latex = True
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':10,'message':"Missing parameters"})
            return res

        if un not in eq:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':11,'message':"The equation does not contain the indicated unknown"})
            return res
        if un.replace(",","").replace(".","").isnumeric() or un == "":
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':13,'message':"Invalid unknown"})
            return res

        un = sp.symbols(un)
        eq_split = eq.split("=")
        if len(eq_split) > 2:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid equality"})
            return res
        try:
            if len(eq_split) == 1:
                eq = sp.sympify(eq_split[0])
            else:
                eq1 = sp.sympify(eq_split[0])
                eq2 = sp.sympify(eq_split[1])
                eq = eq1-eq2
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid equality"})
            return res
        try:
            sol = sp.solve(eq,un)
        except:
            res['statusCode'] = 500
            res['body'] = json.dumps({'problem_id':14,'message':"The service can not resolve this equation"})
            return res

        if not latex:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(input_eq),'solution':str(sol)})
            return res
        else:
            if len(eq_split) == 1:
                input_eq = str(sp.latex(eq))
            else:
                input_eq = str(sp.latex(eq1)) + '=' + str(sp.latex(eq2))
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':input_eq,'solution':str(sp.latex(sol))})
            return res
    except:
        res['statusCode'] = 500
        res['body'] = json.dumps({'problem_id':15,'message':"Generic error while solving equation, check the inserted parameters"})
        return res

In [2]:
event = {}
event['queryStringParameters'] = {}
event['queryStringParameters']['eq'] = "e^x-1"
event['queryStringParameters']['un'] = "x"
event['queryStringParameters']['latex'] = "0"
print(solve_equation(event,None))

{'headers': {'Access-Control-Allow-Origin': '*'}, 'statusCode': 500, 'body': '{"problem_id": 15, "message": "Generic error while solving equation, check the inserted parameters"}'}


# Inequality Solver

In [3]:
import json
import sympy as sp
from sympy.solvers.inequalities import reduce_rational_inequalities

def solve_inequality(event,context):
    res = {}
    res['headers'] = {"Access-Control-Allow-Origin" : "*"}
    try:
        try:
            ineq = event['queryStringParameters']['ineq']
            var = event['queryStringParameters']['var']
            sym = event['queryStringParameters']['sym']
            latex = event['queryStringParameters']['latex']
            
            if latex == "false" or latex == "False" or latex == "0":
                latex = False
            else:
                latex = True
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':10,'message':"Missing parameters"})
            return res

        if var.replace(",","").replace(".","").isnumeric():
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':13,'message':"Invalid variable"})
            return res
        if var not in ineq:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':11,'message':"The equation does not contain the indicated variable"})
            return res
        
        if sym != ">" and sym != ">=" and sym != "<" and sym != "<=":
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':1,'message':"Invalid inequality symbol"})
            return res
        if sym not in ineq and ineq.split(sym) == 2:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':2,'message':"Inequality Symbol not found in the inequality"})
            return res
        
        check_string = ineq.replace(sym,"")
        if len(ineq.split(sym)) > 2 or "=" in check_string or ">" in check_string or ">=" in check_string or "<" in check_string or "<=" in check_string:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid inequality"})
            return res

        var = sp.symbols(var)
        
        if len(ineq.split(sym)) == 1:
            ineq = ineq.split(sym)[0] + str(sym) + "0"
            
        try:
            ineq = sp.sympify(ineq)
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid inequality"})
            return res
        try:
            sol = sp.solve(ineq,var)
        except:
            res['statusCode'] = 500
            res['body'] = json.dumps({'problem_id':14,'message':"The service can not resolve this equation"})
            return res

        if not latex:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(ineq),'solution':str(sol)})
            return res
        else:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(sp.latex(ineq)),'solution':str(sp.latex(sol))})
            return res
    except:
        res['statusCode'] = 500
        res['body'] = json.dumps({'problem_id':15,'message':"Generic error while solving equation, check the inserted parameters"})
        return res

In [4]:
event = {}
event['queryStringParameters'] = {}
event['queryStringParameters']['ineq'] = "ln(x^2) < sin(3)"
event['queryStringParameters']['var'] = "x"
event['queryStringParameters']['sym'] = "<"
event['queryStringParameters']['latex'] = "0"
print(solve_inequality(event,None))

{'headers': {'Access-Control-Allow-Origin': '*'}, 'statusCode': 200, 'body': '{"input": "log(x**2) < sin(3)", "solution": "Ne(x, 0) & (x < exp(sin(3)/2)) & (x > -exp(sin(3)/2))"}'}


# Derivative Calculator

In [7]:
import json
import sympy as sp

def calculate_derivative(event, context):
    res = {}
    res['headers'] = {"Access-Control-Allow-Origin" : "*"}
    try:
        try:
            fun = event['queryStringParameters']['fun']
            var = event['queryStringParameters']['var']
            n = event['queryStringParameters']['n']
            latex = event['queryStringParameters']['latex']

            if latex == "false" or latex == "False" or latex == "0":
                latex = False
            else:
                latex = True
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':10,'message':"Missing parameters"})
            return res
        
        if "=" in fun or "<" in fun or ">" in fun or "<=" in fun or ">=" in fun:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid function"})
            return res
        try:
            fun = sp.sympify(fun)
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid function"})
            return res
        
        if var.replace(",","").replace(".","").isnumeric() or var == "":
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':13,'message':"Invalid variable"})
            return res
        
        if not n.isdigit():
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':1,'message':"Invalid derivative order"})
            return res
        
        var = sp.symbols(var)
        
        try:
            sol = fun.diff((var,n),evaluate=True)
        except:
            res['statusCode'] = 500
            res['body'] = json.dumps({'problem_id':14,'message':"The service can not calculate this derivative"})
            return res

        if not latex:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(fun),'solution':str(sol)})
            return res
        else:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(sp.latex(fun)),'solution':str(sp.latex(sol))})
            return res
    except:
        res['statusCode'] = 500
        res['body'] = json.dumps({'problem_id':15,'message':"Generic error while solving equation, check the inserted parameters"})
        return res

In [8]:
event = {}
event['queryStringParameters'] = {}
event['queryStringParameters']['fun'] = "ln(x^2) + x"
event['queryStringParameters']['var'] = "x"
event['queryStringParameters']['n'] = "1"
event['queryStringParameters']['latex'] = "0"
print(calculate_derivative(event,None))

{'headers': {'Access-Control-Allow-Origin': '*'}, 'statusCode': 200, 'body': '{"input": "x + log(x**2)", "solution": "1 + 2/x"}'}


# Differential Equations Solver

In [41]:
import json
import sympy as sp
from sympy.solvers.inequalities import reduce_rational_inequalities

def solve_differential_equation(event, context):
    res = {}
    res['headers'] = {'Access-Control-Allow-Origin':'*'}
    try:
        try:
            eq = event['queryStringParameters']['eq']
            latex = event['queryStringParameters']['latex']

            if latex == "false" or latex == "False" or latex == "0":
                latex = False
            else:
                latex = True
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':10,'message':"Missing parameters"})
            return res
        
        if "=" in eq or "<" in eq or ">" in eq or "<=" in eq or ">=" in eq:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid equation"})
            return res
        try:
            eq = sp.sympify(eq)
        except:
            res['statusCode'] = 400
            res['body'] = json.dumps({'problem_id':12,'message':"Invalid equation"})
            return res
        
        try:
            sol = sp.dsolve(eq)
        except:
            res['statusCode'] = 500
            res['body'] = json.dumps({'problem_id':14,'message':"The service can not solve this equation"})
            return res

        if not latex:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(eq),'solution':str(sol)})
            return res
        else:
            res['statusCode'] = 200
            res['body'] = json.dumps({'input':str(sp.latex(eq)),'solution':str(sp.latex(sol))})
            return res
    except:
        res['statusCode'] = 500
        res['body'] = json.dumps({'problem_id':15,'message':"Generic error while solving equation, check the inserted parameters"})
        return res

In [43]:
event = {}
event['queryStringParameters'] = {}
event['queryStringParameters']['eq'] = "4*Derivative(y(x),(x,2)) - 2*Derivative(y(x),(x,1)) -3*y(x)"
event['queryStringParameters']['latex'] = "0"
print(solve_differential_equation(event,None)['body'])

{"input": "-3*y(x) - 2*Derivative(y(x), x) + 4*Derivative(y(x), (x, 2))", "solution": "Eq(y(x), C1*exp(x*(1 - sqrt(13))/4) + C2*exp(x*(1 + sqrt(13))/4))"}
